<a href="https://colab.research.google.com/github/oooinr4018-web/-1/blob/main/ESAA_0427_%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 스태킹 앙상블

- 스태킹(Stacking)

: 개별적인 여러 알고리즘을 서로 결합해 예측 결과를 도출 (배깅, 부스팅과의 공통점)

: 개별 알고리즘의 예측 결과 데이터 세트를 최종적인 메타 데이터 세트로 만들어 별도의 ML 알고리즘으로 최종 학습을 수행하고 테스트 데이터를 기반으로 다시 최종 예측을 수행하는 방식

(메타 모델: 개별 모델의 예측된 데이터 세트를 기반으로 하여 학습, 예측하는 방식)

- 스태킹 모델

1. 개별적인 기반 모델

많은 개별 모델 필요 (2~3개의 개별 모델만으로 쉽게 예측 성능 향상 X)

2. 최종 메타 모델: 개별 기반 모델의 예측 데이터를 학습 데이터로 만들어서 학습

-> 여러 개별 모델의 예측 데이터를 각각 스태킹 형태로 결합해 최종 메타 모델의 학습용 피처 데이터 세트, 테스트용 피처 데이터 세트 생성

-> 여러 개의 모델에 대한 예측값을 합한 후 (스태킹 형태로 쌓은 후), 이에 대한 예측을 다시 수행하는 것

- 스태킹 앙상블 모델

M개의 로우, N개의 칼럼(피처) 가진 데이터 세트에 스태킹 앙상블 적용하기

3개의 ML 알고리즘 모델 각각 학습 시키기, 예측 수행 -> M개의 로우를 가진 1개의 레이블 값 도출-> 모델별로 도출된 예측 레이블 값을 다시 합해서 새로운 데이터 세트 생성-> 스태킹된 데이터 세트에 대해 최종적 모델 적용-> 최종 예측

# 기본 스태킹 모델

새로운 주피터 노트북 생성->기본 스태킹 모델을 위스콘신 암 데이터 세트에 적용
-> 학습 데이터 세트와 테스트 데이터 세트로 나누기->스태킹에 사용될 머신러닝 알고리즘 클래스 생성

개별 모델: KNN, 랜덤 포레스트, 결정 트리, 에이다부스트

최종 모델: 로지스틱 회귀

개별 알고리즘으로부터 예측된 예측값을 칼럼 레벨로 옆으로 붙여 피처 값 만들기 -> 로지스틱 회귀에서 학습 데이터로 다시 사용

예측 데이터 세트는 1차원 형태의 ndarray

반환된 예측 결과를 행 형태로 붙이기-> 넘파이의 transpose() 이용해 행, 열 위치 바꾼 ndarray로 변환

-> 최종 메타 모델인 로지스틱 회귀 학습, 예측 정확도 측정

개별 모델의 예측 데이터를 스태킹으로 재구성해 최종 메타 모델에서 학습, 예측한 결과, 정확도가 개별 모델 정확도보다 향상

# CV 세트 기반의 스태킹

과적합 개선을 위해, 교차 검증 기반 예측 결과 데이터 세트 이용

각 모델별로 예측한 결과값을 기반으로 메타 모델을 위한 학습용/테스트용 데이터 생성 -> 개별 모델들이 생성한 학습용 데이터를 스태킹 형태로 합쳐서 최종 학습용 데이터 세트 생성, 최종적 생성된 데이터 세트 예측, 원본 데이터 기반 평가

# HyperOpt 이용한 하이퍼 파라미터 최적화

데이터를 학습용과 테스트용으로 분리

전체 데이터의 80%는 학습, 20%는 테스트로 사용. 학습 데이터 내부에서도 다시 학습용과 검증용으로 나누기

전체적인 흐름

하이퍼파라미터 탐색->모델 학습->성능 평가

(반복 진행 -> 최적의 값 찾기)

- 튜닝할 하이퍼파라미터의 범위 설정

지정된 범위 안에서 hyperopt가 최적값 탐색

- 전달받은 하이퍼파라미터를 이용해 XGBoost 모델 생성

STATUS_OK: 해당 시행이 정상적으로 수행이 되었는지의 여부를 리스트의 status 값에 저장하도록 설정한 것


- 변환 과정

Hyperopt: 하이퍼파라미터 값을 실수 형태로 반환

XGBoost: 정수형 파라미터형태 요구

-> max_depth와 min_child_weight를 int로 변환

- 모델 성능 평가 과정

 cross_val_score -> 교차 검증 수행, 모델의 성능 평가.

 단순 accuracy가 아니라 교차검증을 사용하기 때문에 더 안정적인 성능 측정

Cv=3 코드 -> 3-fold-cross validation (데이터를 3개로 나누어, 데이터를 여러 번 바꾸어가며 평가 진행)

결과를 평균을 내서 최종 정확도 계산

(Hyperopt는 값을 최소화하는 구조인데, 정확도는 클수록 좋은 값이기 때문에 정확도에 -1을 곱해서 값이 클수록 loss가 작아지도록 변환)

fmin함수-> 50번의 시도를 통한 가장 좋은 파라미터 조합 찾기

learning_rate, max_depth, min_child_weight 등의 최적 값 도출 (바로 사용 X)

HyperOpt는 탐색 과정에서 하이퍼파라미터 값을 실수 형태로 반환.

XGBoost에서는 max_depth나 min_child_weight처럼 일부 파라미터는 정수형을 요구.

-> int 함수를 사용해 정수형으로 변환, learning_rate에서 소수점 자리를 정리

- 모델의 학습 성능 향상

n_estimators를 400으로 증가 -> 모델의 학습 성능 향상

(튜닝 단계에서는 속도를 위해 작게 설정, 최종 모델에서는 성능을 위해 크게 설정)

Early stopping -> 성능이 개선되지 않으면 학습 중단

- 결과

정확도: 약 0.9561 (튜닝 이전보다 성능 향상)

-> 하이퍼파라미터 튜닝을 통한 모델 성능 개선

-> 한계점: 데이터 개수(569개)로 비교적 작기 때문에 데이터 분할에 따라 성능 변동이 발생 가능성








In [2]:
!pip install scikit-learn==1.2.2
!pip install xgboost==1.7.6

  Using cached scikit-learn-1.2.2.tar.gz (7.3 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-learn: filename=scikit_learn-1.2.2-cp312-cp312-linux_x86_64.whl size=9452411 sha256=eca8762b0cc0cfeeebf8280c107391a173b1d2d13a063cb9b5f7d2ae59f0b80b
  Stored in directory: /root/.cache/pip/wheels/24/f8/77/ae90c181b806f450a6fec8c8f794594e7c92fa79d7ca27e656
Successfully built scikit-learn
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.2.2 which is incompatible.
mlxtend 0.23.4 requires scikit-learn>=1.3.1, but yo

In [3]:
!pip install lightgbm==3.3.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 30.3 MB/s eta 0:00:00
  Attempting uninstall: lightgbm
    Found existing installation: lightgbm 4.6.0
    Uninstalling lightgbm-4.6.0:
      Successfully uninstalled lightgbm-4.6.0


In [4]:
!pip install numpy==1.26.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 42.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.2.2 which is incompatible.
mlxtend 0.23.4 requires scikit-learn>=1.3.1, but you have scikit-learn 1.2.2 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
spopt 0.7.0 requires scikit-learn>=1.4.0, but you have scikit-learn 1.2.2 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-pytho

In [1]:
import lightgbm
from lightgbm import LGBMClassifier

In [2]:
#LightGBM의 파이썬 패키지인 lightgbm에서 LGBMClassifier 임포트
from lightgbm import LGBMClassifier
from lightgbm import early_stopping, log_evaluation

import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

dataset=load_breast_cancer()

cancer_df=pd.DataFrame(data=dataset.data, columns=dataset.feature_names)
cancer_df['target']=dataset.target
X_features=cancer_df.iloc[:,:-1]
y_label=cancer_df.iloc[:, -1]

# 전체 데이터 중 80%는 학습용 데이터, 20%는 테스트용 데이터 추출
X_train, X_test, y_train, y_test=train_test_split(X_features, y_label, test_size=0.2, random_state=156)

# 위에서 만든 X_train, y_train을 다시 쪼개서 90%는 학습과 10%는 검증용 데이터로 분리
X_tr, X_val, y_tr, y_val=train_test_split(X_train, y_train, test_size=0.1, random_state=156)

# 앞서 XGBoost와 동일하게 n_estimators는 400 설정.
lgbm_wrapper=LGBMClassifier(n_estimators=400, learning_rate=0.05)

# LightGBM도 XGBoost와 동일하게 조기 중단 수행 가능.
evals=[(X_tr, y_tr), (X_val, y_val)]
lgbm_wrapper.fit(X_tr, y_tr, callbacks=[early_stopping(stopping_rounds=50), log_evaluation(period=10)], eval_metric="logloss", eval_set=evals)
preds=lgbm_wrapper.predict(X_test)
pred_proba=lgbm_wrapper.predict_proba(X_test)[:,1]

Training until validation scores don't improve for 50 rounds
[10]	training's binary_logloss: 0.386526	valid_1's binary_logloss: 0.461267
[20]	training's binary_logloss: 0.246443	valid_1's binary_logloss: 0.362062
[30]	training's binary_logloss: 0.167198	valid_1's binary_logloss: 0.310105
[40]	training's binary_logloss: 0.115054	valid_1's binary_logloss: 0.282853
[50]	training's binary_logloss: 0.0789991	valid_1's binary_logloss: 0.267587
[60]	training's binary_logloss: 0.0550801	valid_1's binary_logloss: 0.260746
[70]	training's binary_logloss: 0.0383095	valid_1's binary_logloss: 0.267484
[80]	training's binary_logloss: 0.0264668	valid_1's binary_logloss: 0.270523
[90]	training's binary_logloss: 0.0183664	valid_1's binary_logloss: 0.276485
[100]	training's binary_logloss: 0.0126961	valid_1's binary_logloss: 0.279265
[110]	training's binary_logloss: 0.00882581	valid_1's binary_logloss: 0.282152
Early stopping, best iteration is:
[61]	training's binary_logloss: 0.0532381	valid_1's binary

In [4]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
def get_clf_eval(y_test, pred, pred_proba):
  confusion=confusion_matrix(y_test, pred)
  accuracy=accuracy_score(y_test, pred)
  precision=precision_score(y_test, pred)
  recall=recall_score(y_test, pred)
  f1=f1_score(y_test, pred)
  #ROC_AUC 추가
  roc_auc=roc_auc_score(y_test, pred_proba)
  print('오차행렬')
  print(confusion)
  #ROC_AUC print 추가
  print('정확도:{0:.4f}, 정밀도:{1:.4f}, 재현율{2:.4f}, F1:{3:.4f}, AUC:{4:.4f}'.format(accuracy, precision, recall, f1, roc_auc))


In [5]:
get_clf_eval(y_test, preds, pred_proba)

오차행렬
[[34  3]
 [ 2 75]]
정확도:0.9561, 정밀도:0.9615, 재현율0.9740, F1:0.9677, AUC:0.9877


In [ ]:
# plot_importance()를 이용하여 feature 중요도 시각화
from lightgbm import plot_importance
import matplotlib.pyplot as plt
%matplotlib inline

fig, ax=plt.subplots(figsize=(10,12))
plot_importance(lgbm_wrapper, ax=ax)

In [6]:
params={
'max_depth': [10,20,30,40,50], 'num_leaves': [35,45,55,65],
'colsample_bytree': [0.5, 0.6, 0.7, 0.8, 0.9], 'subsample': [0.5, 0.6, 0.7, 0.8, 0.9],
'min_child_weight': [10,20,30,40], 'reg_alpha' : [0.01, 0.05, 0.1]
}

In [7]:
pip install hyperopt

In [8]:
from hyperopt import hp

#-10~10까지 1간격을 가지는 입력 변수 x와 -15~15까지 1간격으로 입력 변수 y 설정.
search_space={'x':hp.quniform('x', -10, 10, 1), 'y':hp.quniform('y', -15, 15, 1)}


In [9]:
from hyperopt import STATUS_OK

# 목적 함수를 생성, 변숫값과 변수 검색 공간을 가지는 딕셔너리를 인자로 받고, 특정 값을 반환
def objective_func(search_space):
  x=search_space['x']
  y=search_space['y']
  retval=x**2-20*y

  return retval

In [10]:
from hyperopt import fmin, tpe, Trials
# 입력 결괏값을 저장한 Trials 객체값 생성.
trial_val=Trials()

# 목적 함수의 최솟값을 반환하는 최적 입력 변숫값을 5번의 입력값 시도(max_eval=5)로 찾아냄.
best_01=fmin(fn=objective_func, space=search_space, algo=tpe.suggest, max_evals=5,
             trials=trial_val, rstate=np.random.default_rng(seed=0))
print('best:', best_01)



100%|██████████| 5/5 [00:00<00:00, 452.77trial/s, best loss: -224.0]
best: {'x': -4.0, 'y': 12.0}


In [11]:
trial_val=Trials()

#max_evals를 20회로 늘려서 재테스트
best_02=fmin(fn=objective_func, space=search_space, algo=tpe.suggest, max_evals=20,
             trials=trial_val, rstate=np.random.default_rng(seed=0))
print('best:', best_02)

100%|██████████| 20/20 [00:00<00:00, 411.82trial/s, best loss: -296.0]
best: {'x': 2.0, 'y': 15.0}


In [12]:
# fmin()에 인자로 들어가는 Trials 객체의 result 속성에 파이썬 리스트로 목적 함수 반환값들이 저장됨
# 리스트 내부의 개별 원소는 {'loss': 함수 반환값, 'status': 반환 상태값}와 같은 딕셔너리임.
print(trial_val.results)

[{'loss': -64.0, 'status': 'ok'}, {'loss': -184.0, 'status': 'ok'}, {'loss': 56.0, 'status': 'ok'}, {'loss': -224.0, 'status': 'ok'}, {'loss': 61.0, 'status': 'ok'}, {'loss': -296.0, 'status': 'ok'}, {'loss': -40.0, 'status': 'ok'}, {'loss': 281.0, 'status': 'ok'}, {'loss': 64.0, 'status': 'ok'}, {'loss': 100.0, 'status': 'ok'}, {'loss': 60.0, 'status': 'ok'}, {'loss': -39.0, 'status': 'ok'}, {'loss': 1.0, 'status': 'ok'}, {'loss': -164.0, 'status': 'ok'}, {'loss': 21.0, 'status': 'ok'}, {'loss': -56.0, 'status': 'ok'}, {'loss': 284.0, 'status': 'ok'}, {'loss': 176.0, 'status': 'ok'}, {'loss': -171.0, 'status': 'ok'}, {'loss': 0.0, 'status': 'ok'}]


In [13]:
# Trials 객체의 vals 속성에 {'입력변수명': 개별 수행 시마다 입력된 값 리스트} 형태로 저장됨.
print(trial_val.vals)

{'x': [-6.0, -4.0, 4.0, -4.0, 9.0, 2.0, 10.0, -9.0, -8.0, -0.0, -0.0, 1.0, 9.0, 6.0, 9.0, 2.0, -2.0, -4.0, 7.0, -0.0], 'y': [5.0, 10.0, -2.0, 12.0, 1.0, 15.0, 7.0, -10.0, 0.0, -5.0, -3.0, 2.0, 4.0, 10.0, 3.0, 3.0, -14.0, -8.0, 11.0, -0.0]}


In [14]:
import pandas as pd

# results에서 loss 키값에 해당하는 벨류들을 추출하여 list로 생성.
losses=[loss_dict['loss'] for loss_dict in trial_val.results]

# DataFrame으로 생성.
result_df=pd.DataFrame({'x': trial_val.vals['x'], 'y': trial_val.vals['y'], 'losses':losses})
result_df

,x,y,losses
0,-6.0,5.0,-64.0
1,-4.0,10.0,-184.0
2,4.0,-2.0,56.0
3,-4.0,12.0,-224.0
4,9.0,1.0,61.0
5,2.0,15.0,-296.0
6,10.0,7.0,-40.0
7,-9.0,-10.0,281.0
8,-8.0,0.0,64.0
9,-0.0,-5.0,100.0


In [15]:
# 전체 데이터 중 80%는 학습용 데이터, 20%는 테스트용 데이터 추출
X_train, X_test, y_train, y_test=train_test_split(X_features, y_label, test_size=0.2, random_state=156)

# 앞에서 추출한 학습 데이터를 다시 학습과 검증 데이터로 분리
X_tr, X_Val, y_tr, y_val=train_test_split(X_train, y_train, test_size=0.1, random_state=156)


In [16]:
from hyperopt import hp

# max_depth는 5에서 20까지 1간격으로, min_child_weight는 1에서 2까지 1간격으로
# colsample_bytree는 0.5에서 1사이, learning_rate는 0.01애서 0.2 사이 정규 분포된 값으로 검색,
xgb_search_space={'max_depth': hp.quniform('max_depth', 5, 20, 1),
                  'min_child_weight': hp.quniform('min_child_weight', 1, 2, 1),
                  'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),
                  'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1),
                  }

In [17]:
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier
from hyperopt import STATUS_OK

#fmin()에서 입력된 search_space 값으로 입력된 모든 값은 실수형임.
#XGBClassifier의 정수형 하이퍼 파라미터는 정수형 변환을 해줘야 함.
# 정확도는 높을수록 더 좋은 수치임. -1*정확도를 곱해서 큰 정확도 값일수록 최소가 되도록 변환
def objective_func(search_space):
  # 수행 시간 절약을 위해 nestimators는 100으로 축소
  xgb_clf=XGBClassifier(n_estimator=100, max_depth=int(search_space['max_depth']),
                        min_child_weight=int(search_space['min_child_weight']),
                        learning_rate=search_space['learning_rate'],
                        colsample_bytree=search_space['colsample_bytree'],
                        eval_metric='logloss')
  accuracy=cross_val_score(xgb_clf, X_train, y_train, scoring='accuracy', cv=3)

  # accuracy는 cv=3 개수만큼 roc-auc 결과를 리스트로 가짐. 이를 평균해서 반환하되 -1을 곱함.
  return {'loss':-1*np.mean(accuracy), 'status': STATUS_OK}

In [18]:
from hyperopt import fmin, tpe, Trials

trial_val=Trials()
best=fmin(fn=objective_func,
          space=xgb_search_space,
          algo=tpe.suggest,
          max_evals=50, # 최대 반복 횟수를 지정합니다.
          trials=trial_val, rstate=np.random.default_rng(seed=9))
print('best:', best)

[08:38:41] WARNING: ../src/learner.cc:767: 
Parameters: { "n_estimator" } are not used.

[08:38:43] WARNING: ../src/learner.cc:767: 
Parameters: { "n_estimator" } are not used.

[08:38:45] WARNING: ../src/learner.cc:767: 
Parameters: { "n_estimator" } are not used.

[08:38:46] WARNING: ../src/learner.cc:767: 
Parameters: { "n_estimator" } are not used.

[08:38:46] WARNING: ../src/learner.cc:767: 
Parameters: { "n_estimator" } are not used.

[08:38:46] WARNING: ../src/learner.cc:767: 
Parameters: { "n_estimator" } are not used.

[08:38:47] WARNING: ../src/learner.cc:767: 
Parameters: { "n_estimator" } are not used.

[08:38:47] WARNING: ../src/learner.cc:767: 
Parameters: { "n_estimator" } are not used.

[08:38:47] WARNING: ../src/learner.cc:767: 
Parameters: { "n_estimator" } are not used.

[08:38:48] WARNING: ../src/learner.cc:767: 
Parameters: { "n_estimator" } are not used.

[08:38:48] WARNING: ../src/learner.cc:767: 
Parameters: { "n_estimator" } are not used.

[08:38:48] WARNING: .

In [19]:
print('colsample_bytree:{0}, learning_rate:{1}, max_depth:{2}, min_child_weight:{3}'.format(
    round(best['colsample_bytree'], 5), round(best['learning_rate'], 5),
    int(best['max_depth']), int(best['min_child_weight'])))

colsample_bytree:0.95994, learning_rate:0.1548, max_depth:6, min_child_weight:2


In [20]:
xgb_wrapper=XGBClassifier(n_estimator=400,
                          learning_rate=round(best['learning_rate'], 5),
                          max_depth=int(best['max_depth']),
                          min_child_weight=int(best['min_child_weight']),
                          colsample_bytree=round(best['colsample_bytree'], 5)
                          )
evals=[(X_tr, y_tr), (X_val, y_val)]
xgb_wrapper.fit(X_tr, y_tr, early_stopping_rounds=50, eval_metric='logloss',
                eval_set=evals, verbose=True)

preds=xgb_wrapper.predict(X_test)
pred_proba=xgb_wrapper.predict_proba(X_test)[:,1]

get_clf_eval(y_test, preds, pred_proba)

[08:39:12] WARNING: ../src/learner.cc:767: 
Parameters: { "n_estimator" } are not used.

[0]	validation_0-logloss:0.56834	validation_1-logloss:0.60660
[1]	validation_0-logloss:0.47552	validation_1-logloss:0.54538
[2]	validation_0-logloss:0.40208	validation_1-logloss:0.48735
[3]	validation_0-logloss:0.34468	validation_1-logloss:0.45698
[4]	validation_0-logloss:0.29775	validation_1-logloss:0.41729
[5]	validation_0-logloss:0.26004	validation_1-logloss:0.39167
[6]	validation_0-logloss:0.22681	validation_1-logloss:0.36682
[7]	validation_0-logloss:0.20096	validation_1-logloss:0.34593
[8]	validation_0-logloss:0.17762	validation_1-logloss:0.33030
[9]	validation_0-logloss:0.15762	validation_1-logloss:0.31918
[10]	validation_0-logloss:0.14233	validation_1-logloss:0.30772
[11]	validation_0-logloss:0.12769	validation_1-logloss:0.30104
[12]	validation_0-logloss:0.11566	validation_1-logloss:0.29621
[13]	validation_0-logloss:0.10479	validation_1-logloss:0.29157
[14]	validation_0-logloss:0.09640	valid

/usr/local/lib/python3.12/dist-packages/xgboost/sklearn.py:835: UserWarning: `eval_metric` in `fit` method is deprecated for better compatibility with scikit-learn, use `eval_metric` in constructor or`set_params` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/xgboost/sklearn.py:835: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


[52]	validation_0-logloss:0.02071	validation_1-logloss:0.26869
[53]	validation_0-logloss:0.02055	validation_1-logloss:0.27083
[54]	validation_0-logloss:0.02040	validation_1-logloss:0.27105
[55]	validation_0-logloss:0.02026	validation_1-logloss:0.27354
[56]	validation_0-logloss:0.02013	validation_1-logloss:0.27299
[57]	validation_0-logloss:0.02000	validation_1-logloss:0.27293
[58]	validation_0-logloss:0.01986	validation_1-logloss:0.27131
[59]	validation_0-logloss:0.01972	validation_1-logloss:0.27341
[60]	validation_0-logloss:0.01960	validation_1-logloss:0.27364
[61]	validation_0-logloss:0.01948	validation_1-logloss:0.27206
[62]	validation_0-logloss:0.01935	validation_1-logloss:0.27347
[63]	validation_0-logloss:0.01923	validation_1-logloss:0.27544
[64]	validation_0-logloss:0.01912	validation_1-logloss:0.27390
[65]	validation_0-logloss:0.01900	validation_1-logloss:0.27140
[66]	validation_0-logloss:0.01889	validation_1-logloss:0.27092
[67]	validation_0-logloss:0.01878	validation_1-logloss:

In [22]:
import numpy as np

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

cancer_data=load_breast_cancer()

X_data=cancer_data.data
y_label=cancer_data.target

X_train, X_test, y_train, y_test=train_test_split(X_data, y_label, test_size=0.2, random_state=0)

In [24]:
# 개별 ML 모델 생성
knn_clf=KNeighborsClassifier(n_neighbors=4)
rf_clf=RandomForestClassifier(n_estimators=100, random_state=0)
df_clf=DecisionTreeClassifier()
ada_clf=AdaBoostClassifier(n_estimators=100)

# 스태킹으로 만들어진 데이터 세트를 학습, 예측할 최종 모델
lr_final=LogisticRegression()

In [25]:
# 개별 모델들을 학습.
knn_clf.fit(X_train, y_train)
rf_clf.fit(X_train, y_train)
df_clf.fit(X_train, y_train)
ada_clf.fit(X_train, y_train)

AdaBoostClassifier(n_estimators=100)

In [31]:
# 학습된 개별 모델들이 각자 변환하는 예측 데이터 세트를 생성하고 개별 모델의 정확도 측정.
knn_pred=knn_clf.predict(X_test)
rf_pred=rf_clf.predict(X_test)
dt_pred=df_clf.predict(X_test)
ada_pred=ada_clf.predict(X_test)
print('KNN 정확도: {0:.4f}'.format(accuracy_score(y_test, knn_pred)))
print('랜덤 포레스트 정확도:{0:4f}'.format(accuracy_score(y_test, rf_pred)))
print('결정 트리 정확도:{0:4f}'.format(accuracy_score(y_test, df_pred)))
print('에이다부스트 정확도: {0:.4f}'.format(accuracy_score(y_test, ada_pred)))

KNN 정확도: 0.9211
랜덤 포레스트 정확도:0.964912
결정 트리 정확도:0.912281
에이다부스트 정확도: 0.9561


In [32]:
pred=np.array([knn_pred, rf_pred, dt_pred, ada_pred])
print(pred.shape)

# transpose를 이용해 행과 열의 위치 교환, 칼럼 레벨로 각 알고리즘의 예측 결과를 피처로 만듦.
pred=np.transpose(pred)
print(pred.shape)

(4, 114)
(114, 4)


In [33]:
lr_final.fit(pred, y_test)
final=lr_final.predict(pred)

print('최종 메타 모델의 예측 정확도:{0:4f}'.format(accuracy_score(y_test, final)))


최종 메타 모델의 예측 정확도:0.973684


In [42]:
# 개별 기반 모델에서 최종 메타 모델이 사용할 학습 및 테스트용 데이터를 생성하기 위한 함수.
def get_stacking_base_datasets(model, X_train_n, y_train_n, X_test_n, n_folds):
  # 지정된 n_folds값으로 kFold 생성.
  kf=KFold(n_splits=n_folds, shuffle=False)
  # 추후에 메타 모델이 사용할 학습 데이터 반환을 위한 넘파이 배열 초기화
  train_fold_pred=np.zeros((X_train_n.shape[0],1))
  test_pred=np.zeros((X_test_n.shape[0], n_folds))
  print(model.__class__.__name__,'model 시작')
  X_tr=X_train_n[train_index]
  y_tr=y_train_n[train_index]
  X_te=X_train_n[test_index]

  # 폴드 세트 내부에서 다시 만들어진 학습 데이터로 기반 모델의 학습 수행.
  model.fit(X_tr, y_tr)
  # 폴드 세트 내부에서 다시 만들어진 검증 데이터로 기반 모델 예측 후 데이터 저장.
  train_fold_pred[valid_index, :]=model.predict(X_te).reshape(-1,1)
  # 입력된 원본 테스트 데이터를 폴드 세트내 학습된 기반 모델에서 예측 후 데이터 저장.
  test_pred[:, foldeer_counter]=model.predict(X_test_n)

  # 폴드 세트 내에서 원본 테스트 데이터를 예측한 데이터를 평균하여 테스트 데이터로 생성
  test_pred_mean=np.mean(test_pred, axis=1).reshape(-1,1)

  #train_fold_pred는 최종 메타 모델이 사용하는 학습 데이터, test_pred_mean은 테스트 데이터
  return train_fold_pred, test_pred_mean

In [45]:
from sklearn.model_selection import KFold
knn_train, knn_test=get_stacking_base_datasets(knn_clf, X_train, y_train, X_test, 7)
rf_train, rf_test=get_stacking_base_datasets(rf_clf, X_train, y_train, X_test, 7)
dt_train, dt_test=get_stacking_base_datasets(dt_clf, X_train, y_train, X_test, 7)
ada_train, ada_test=get_stacking_base_datasets(ada_clf, X_train, y_train, X_test, 7)

KNeighborsClassifier model 시작


NameError: name 'train_index' is not defined

In [38]:
Stack_final_X_train=np.concatenate((knn_train, rf_train, dt_train, ada_train), axis=1)
Stack_final_X_test=np.concatenate((knn_test, rf_test, dt_test, ada_test), axis=1)
print('원본 학습 피처 데이터 Shape:',  X_train.shape, '원본 테스트 피처 노멛:', X_test.shape)
print('스태킹 학습 피처 데이터 Shape:', Stack_final_X_train.shape,
      '스태킹 테스트 피처 데이터 Shape:', Stack_final_X_test.shape)

NameError: name 'knn_train' is not defined

In [39]:
lr_final.fit(Stack_final_X_train, y_train)
stack_final=lr_final.predict(Stack_final_X_test)

print('최종 메타 모델의 예측 정확도: {0:.4f}'.format(accuracy_score(y_test, stack_final)))

NameError: name 'Stack_final_X_train' is not defined